In [1]:
import os
import random
import json
import shutil
from tqdm import tqdm

# Paths
SOURCE_DIR = "../../../datasets/chosen_images"
OUTPUT_ROOT = "../../../datasets/RealBubbles.coco-segmentation"
INPUT_JSON = "result_coco.json"

In [2]:
# Settings
SPLIT_RATIOS = {"train": 0.7, "val": 0.2, "test": 0.1}
SEED = 42

# Load input COCO-style annotations
with open(INPUT_JSON, "r") as f:
    data = json.load(f)

# Fix image paths: extract only filename from full path
for img in data["images"]:
    full_path = img["file_name"]
    filename = os.path.basename(full_path)
    img["file_name"] = filename

# Get full list of filenames from COCO JSON that exist in SOURCE_DIR
available_filenames = set(os.listdir(SOURCE_DIR))
coco_images = [img for img in data["images"] if img["file_name"] in available_filenames]

# Reindex image IDs and build mapping
image_id_map = {}
for new_id, img in enumerate(coco_images, 1):
    old_id = img["id"]
    img["id"] = new_id
    image_id_map[old_id] = new_id

# Filter annotations to match the selected images
image_ids_selected = set(image_id_map.keys())
filtered_anns = [
    {**ann, "image_id": image_id_map[ann["image_id"]], "id": i + 1}
    for i, ann in enumerate(data["annotations"])
    if ann["image_id"] in image_ids_selected
]

from collections import defaultdict

# === Group images by prefix ===
prefix_groups = defaultdict(list)
for img in coco_images:
    filename = img["file_name"]
    prefix = filename.rsplit("_", 1)[0]  # e.g., "prefix1" from "prefix1_00001"
    prefix_groups[prefix].append(img)

# === Split per prefix group ===
total_images = sum(len(imgs) for imgs in prefix_groups.values())
target_train = int(SPLIT_RATIOS["train"] * total_images)
target_val   = int(SPLIT_RATIOS["val"] * total_images)
target_test  = total_images - target_train - target_val

splits = {"train": [], "val": [], "test": []}

for prefix, images in prefix_groups.items():
    random.seed(SEED)
    random.shuffle(images)

    # Temporarily assign based on ratios
    n_total = len(images)
    frac_train = round(SPLIT_RATIOS["train"] * n_total)
    frac_val   = round(SPLIT_RATIOS["val"] * n_total)
    frac_test  = n_total - frac_train - frac_val

    splits["train"].extend(images[:frac_train])
    splits["val"].extend(images[frac_train:frac_train + frac_val])
    splits["test"].extend(images[frac_train + frac_val:])


# Re-shuffle within each final split (optional)
for split in splits:
    random.shuffle(splits[split])


# Create output directories and split-specific annotation files
for split_name, split_images in splits.items():
    print(f"\nProcessing split: {split_name} ({len(split_images)} images)")
    split_dir = os.path.join(OUTPUT_ROOT, split_name)
    os.makedirs(split_dir, exist_ok=True)

    split_image_ids = {img["id"] for img in split_images}
    split_anns = [ann for ann in filtered_anns if ann["image_id"] in split_image_ids]

    # Copy images
    for img in tqdm(split_images):
        src_path = os.path.join(SOURCE_DIR, img["file_name"])
        dst_path = os.path.join(split_dir, img["file_name"])
        shutil.copy2(src_path, dst_path)

    # Clean up: remove 'path' key if present
    for img in split_images:
        img.pop("path", None)

    # 🔧 Add "info" and optional "licenses" fields
    out_json = {
        "info": {
            "description": f"{split_name} split of bubble dataset",
            "version": "1.0",
            "year": 2025,
            "contributor": "Jorge Costa",
            "date_created": "2025-06-19"
        },
        "licenses": [{
            "id": 1,
            "name": "CC-BY-4.0",
            "url": "https://creativecommons.org/licenses/by/4.0/"
        }],
        "images": split_images,
        "annotations": split_anns,
        "categories": data["categories"]
    }

    # Write to file
    out_json_path = os.path.join(split_dir, "_annotations.coco.json")
    with open(out_json_path, "w") as f:
        json.dump(out_json, f, indent=2)

print("\n✅ Done! COCO-style splits created in:")
for s in splits:
    print(f"  - {os.path.join(OUTPUT_ROOT, s)}")



Processing split: train (171 images)


100%|██████████| 171/171 [00:04<00:00, 34.87it/s]



Processing split: val (48 images)


100%|██████████| 48/48 [00:01<00:00, 34.70it/s]



Processing split: test (25 images)


100%|██████████| 25/25 [00:00<00:00, 33.99it/s]



✅ Done! COCO-style splits created in:
  - ../../../datasets/RealBubbles.coco-segmentation/train
  - ../../../datasets/RealBubbles.coco-segmentation/val
  - ../../../datasets/RealBubbles.coco-segmentation/test
